<a href="https://colab.research.google.com/github/patrickbryant1/EvoBind-multimer/blob/main/EvoBind_multimer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#EvoBind-multimer
EvoBind-multimer (EBM) designs peptide molecular glues towards user-specified target residues - or **completely blind** - using only perotein sequence information.

EvoBind-multimer accounts for adaptation of the receptor interface structure to the peptide design during optimisation. This consideration of flexibility is crucial for binding.

EvoBind-multimer is an adaptation of EvoBind, based on AlphaFold2, which is available under the [Apache License, Version 2.0](http://www.apache.org/licenses/LICENSE-2.0).  \
The AlphaFold2 parameters are made available under the terms of the [CC BY 4.0 license](https://creativecommons.org/licenses/by/4.0/legalcode) and have not been modified.
\
The design protocol EvoBind2 is made available under the terms of the [CC BY-NC 4.0 license](https://creativecommons.org/licenses/by-nc/4.0/). \
**You may not use these files except in compliance with the licenses.**

## Local installation
For local installation of EvoBind-multimer see: Setup in the repository (https://github.com/patrickbryant1/EvoBind-multimer.git)

If you like EvoBind-multimer - **please star the repo!**

## Citation
If you use EvoBind-multimer, please cite:
<Preprint>
[Brunner A., Wierbilowicz K., Daumiller D., Li Q., Karlsson K., Sangfelt O. and Bryan P. De novo design of macrocyclic molecular glues from protein sequences. bioRxiv 2026.06..; doi:]()

In [4]:
#@title Install dependencies
#@markdown Make sure your runtime is GPU.
#@markdown In the menu above do: Runtime --> Change runtime type --> Hardware accelerator (set to GPU)

#@markdown **Press play.**

#@markdown Simply press play on each cell below and follow the instructions.
#@markdown The installation takes a few minutes.

#@markdown After it finishes (the play button wheel stops spinning) do: Runtime > Restart session (above).

!pip install -q --no-warn-conflicts dm-haiku==0.0.11
!pip install -q --no-warn-conflicts ml-collections
!pip install -q --no-warn-conflicts biopython==1.81
!pip install -q --no-warn-conflicts chex==0.1.5
!pip install -q --no-warn-conflicts dm-tree==0.1.8
!pip install -q --no-warn-conflicts immutabledict==2.0.0
# Removed strict version constraints below:
!pip install -q --no-warn-conflicts scipy
!pip install -q --no-warn-conflicts tensorflow
!pip install -q --no-warn-conflicts rdkit  # Changed from rdkit-pypi
!pip install -q --no-warn-conflicts py3Dmol
!pip install -q --no-warn-conflicts numpy
!pip uninstall -y jax jaxlib
!pip install -q  --no-warn-conflicts jaxlib==0.4.35
!pip install -q --no-warn-conflicts 'jax[cuda12_pip]'==0.4.35 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 102.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 9.3 MB/s eta 0:00:00
Found existing installation: jax 0.11.1
Uninstalling jax-0.11.1:
  Successfully uninstalled jax-0.11.1
Found existing installation: jaxlib 0.11.1
Uninstalling jaxlib-0.11.1:
  Successfully uninstalled jaxlib-0.11.1


In [ ]:
#@title Clone the EvoBind github repo
#@markdown Again, once the box above finshes running, make sure to restart the session (Runtime > Restart session)!
from google.colab import userdata
import os

# 1. WIPE THE OLD DIRECTORY IF IT EXISTS
if os.path.exists('/content/EvoBind-multimer'):
    !rm -rf /content/EvoBind-multimer
    print("Old EvoBind-multimer folder wiped! Ready for the fresh code.")

# Retrieve the token from Colab Secrets
github_token = userdata.get('Github_colab')

# Construct the clone URL securely
repo_url = f"https://{github_token}@github.com/patrickbryant1/EvoBind-multimer.git"

# Clone the repository
!git clone {repo_url}

Cloning into 'EvoBind-multimer'...
remote: Enumerating objects: 649, done.
remote: Counting objects: 100% (649/649), done.
remote: Compressing objects: 100% (434/434), done.
remote: Total 649 (delta 200), reused 638 (delta 192), pack-reused 0 (from 0)
Receiving objects: 100% (649/649), 28.29 MiB | 20.39 MiB/s, done.
Resolving deltas: 100% (200/200), done.


In [5]:
#@title #Follow all steps outlined below to design a binder.
#@markdown To try the **test case** [VHL-KRAS](), press the play button to the left.
\
#@markdown If you don't want to run the test case, **change the input parameters**.

#@markdown #Parameters
#@markdown - *TARGET_1_CHAIN* - what chain in the PDB file to use as target 1 (i.e. VHL)
#@markdown - *TARGET_2_CHAIN* - what chain in the PDB file to use as target 2 (i.e. KRAS)
#@markdown - *TARGET_1* - amino acid sequence of the target 1 chain
#@markdown - *TARGET_2* - amino acid sequence of the target 2 chain
#@markdown - **Optional**: *TARGET_RESIDUES_1* - what residue numbers in the target 1 sequence to design a binder towards. Separated by commas.
#@markdown If not provided, EvoBind multimer will pick a binding spot.
#@markdown - **Optional**: *TARGET_RESIDUES_2* - what residue numbers in the target 2 sequence to design a binder towards. Separated by commas.
#@markdown If not provided, EvoBind multimer will pick a binding spot.
#@markdown - *GLUE LENGTH* - how many amino acids in the peptide (i.e. 8, max 19 here)
#@markdown - **Optional**: *START_PEPTIDE_SEQUENCE* - sequence to start the optimisation from. If no sequence is provided, a random sequence is initialised. If you don't have a good start sequence, leave this empty. If you want to simply predict a protein-peptide interaction, input a start sequence and set NITER to 1, then download the result below.
#@markdown - *NITER: Number of iterations* - how many iterations to optimise (default=200)
#@markdown - **Optional:** CYCLIC_OFFSET - design a cyclic peptide binder.
#@markdown - **TARGET 1 MSA** - currently no MSA search is available directly in this notebook, therefore you have to provide your own MSA in a3m format and upload it here. \
#@markdown - **TARGET 2 MSA** - currently no MSA search is available directly in this notebook, therefore you have to provide your own MSA in a3m format and upload it here. \
#@markdown There are two ways of doing this: \
#@markdown 1. Search uniclust_30 locally with HHblits \
#@markdown 2. Go to https://toolkit.tuebingen.mpg.de/tools/hhblits \
#@markdown Paste the receptor sequence in the search field in fasta format --> Submit. \
#@markdown When the search is finished, go to the tab "Query MSA" and "Download Full A3M" \
#@markdown - Upload the MSA here: \
#@markdown Click the folder icon (Files) to the left and select the upload file icon. Upload the .a3m file.
#@markdown Make sure the MSA is named **TARGET_***.a3m, where PDBID is the PDBID specified above.
import sys, os, csv
from google.colab import files
import pandas as pd
import numpy as np
import urllib.request
import py3Dmol
import matplotlib.pyplot as plt
import glob
import warnings
from Bio.PDB.PDBExceptions import PDBConstructionWarning
# Mute the discontinuous chain warnings
warnings.simplefilter('ignore', PDBConstructionWarning)

sys.path.insert(0,'/content/EvoBind-multimer/src')

job_name = 'test_VHL_KRAS' #@param {type:"string"}
#PDBID = "8QU8" #@param {type:"string"}
#UPLOAD_PDB = True # @param {type:"boolean"}
OUTDIR="/content/"+job_name+'/'
#Make outdir
if not os.path.exists(OUTDIR):
  os.mkdir(OUTDIR)

##PARAMETERS##
TARGET_1= "VHL" #@param {type:"string"}
TARGET_1_CHAIN = "A" #@param {type:"string"}
TARGET_1_SEQ = "VNSREPSQVIFCNRSPRVVLPVWLNFDGEPQPYPTLPPGTGRRIHSYRGHLWLFRDAGTHDGLLVNQTELFVPSLNVDGQPIFANIT" #@param {type:"string"}
TARGET_RESIDUES_1 = "" #@param {type:"string"}
if len(TARGET_RESIDUES_1)<1:
  TARGET_RESIDUES_1=np.arange(len(TARGET_1_SEQ))
else:
  TARGET_RESIDUES_1 = [int(x) for x in TARGET_RESIDUES_1.split(',')]
TARGET_1_MSA = "VHL.a3m" #@param {type:"string"}

TARGET_2= "KRAS" #@param {type:"string"}
TARGET_2_CHAIN = "B" #@param {type:"string"}
TARGET_2_SEQ = "GMTEYKLVVVGAVGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVIDGETCLLDILDTAGQEEYSAMRDQYMRTGEGFLCVFAINNTKSFEDIHHYREQIKRVKDSEDVPMVLVGNKSDLPSRTVDTKQAQDLARSYGIPFIETSAKTRQGVDDAFYTLVREIRKHKEK" #@param {type:"string"}
TARGET_RESIDUES_2 = "" #@param {type:"string"}
if len(TARGET_RESIDUES_2)<1:
  TARGET_RESIDUES_2=np.arange(len(TARGET_2_SEQ))
else:
  TARGET_RESIDUES_2 = [int(x) for x in TARGET_RESIDUES_2.split(',')]
TARGET_2_MSA = "KRAS.a3m" #@param {type:"string"}

GLUE_LENGTH =  8#@param {type:"integer"}
START_GLUE_SEQUENCE = "SKNQPPSP" #@param {type:"string"}
NITER =  10#@param {type:"integer"}
CYCLIC_OFFSET = True # @param {type:"boolean"}


#Try .pdb
if job_name=='test_VHL_KRAS':
    STRUCTURE='/content/EvoBind-multimer/data/test/VHL_KRAS/8QU8.pdb'
    #Read and display the PDB file
    sys.path.insert(0,'/content/EvoBind-multimer/src/')
    from prepare_input_colab import prepare_input
    #Get the protein 1 CAs
    TARGET_1_CAs, TARGET_1_PDBSEQ = prepare_input(STRUCTURE, TARGET_1_CHAIN, TARGET_RESIDUES_1, OUTDIR)
    #Get the protein 2 CAs
    TARGET_2_CAs, TARGET_2_PDBSEQ = prepare_input(STRUCTURE, TARGET_2_CHAIN, TARGET_RESIDUES_2, OUTDIR)

    #Print the receptor PDB sequence
    print('The protein target 1 sequence according to the specified PDB file is:',TARGET_1_PDBSEQ)
    print('The protein target 2 sequence according to the specified PDB file is:',TARGET_2_PDBSEQ)
    print('Make sure this is the sequences you are using for the MSA as well.')
    #Check the sequence and structure
    if TARGET_1_SEQ!=TARGET_1_PDBSEQ:
      print('ERROR! The specified sequence 1 and the sequence in the PDB file are not identical.')
      print('Specified sequence 1:', TARGET_1_SEQ)
      print('PDB sequence 1', TARGET_1_PDBSEQ)
    elif TARGET_2_SEQ!=TARGET_2_PDBSEQ:
      print('ERROR! The specified sequence 2 and the sequence in the PDB file are not identical.')
      print('Specified sequence 2:', TARGET_2_SEQ)
      print('PDB sequence 2', TARGET_2_PDBSEQ)

    view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js',)
    # Check .pdb or a .cif
    file_format = 'pdb' if STRUCTURE.endswith('.pdb') else 'cif'
    view.addModel(open(STRUCTURE, 'r').read(), file_format)
    view.setStyle({}, {})
    view.setStyle({'chain': TARGET_1_CHAIN},{'cartoon': {'color':'green'}})
    view.setStyle({'chain': TARGET_2_CHAIN},{'cartoon': {'color':'cyan'}})
    view.zoomTo()
    view.show()


# write fastas from seqs --------------------------------------------------
if job_name=='test_VHL_KRAS':
  TARGET_1_FASTA = '/content/EvoBind-multimer/data/test/VHL_KRAS/VHL.fasta'
  TARGET_2_FASTA = '/content/EvoBind-multimer/data/test/VHL_KRAS/KRAS.fasta'
elif job_name=='test_VHL_BRD4':
  TARGET_1_FASTA = '/content/EvoBind-multimer/data/test/VHL_BRD4/VHL.fasta'
  TARGET_2_FASTA = '/content/EvoBind-multimer/data/test/VHL_BRD4/BRD4.fasta'
else:
  # Write individual FASTA files
    TARGET_1_FASTA = OUTDIR + TARGET_1 + '.fasta'
    with open(TARGET_1_FASTA, 'w') as file:
        file.write('>' + TARGET_1 + ' \n' + TARGET_1_SEQ)

    TARGET_2_FASTA = OUTDIR + TARGET_2 + '.fasta'
    with open(TARGET_2_FASTA, 'w') as file:
        file.write('>' + TARGET_2 + ' \n' + TARGET_2_SEQ)

    # Create a CSV for efficient INDIVIDUAL MSA generation
    single_queries_path = OUTDIR + "single_queries.csv"
    with open(single_queries_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['id', 'sequence'])
        writer.writerow([TARGET_1, TARGET_1_SEQ])
        writer.writerow([TARGET_2, TARGET_2_SEQ])

# MSAs ----------------------------------------------------------------------
if job_name=='test_VHL_KRAS':
    TARGET_1_MSA = '/content/EvoBind-multimer/data/test/VHL_KRAS/' + TARGET_1_MSA
    TARGET_2_MSA = '/content/EvoBind-multimer/data/test/VHL_KRAS/' + TARGET_2_MSA
    PAIRED_MSA = '/content/EvoBind-multimer/data/test/VHL_KRAS/VHL_KRAS_paired.a3m'
    BLOCKED_MSA = '/content/EvoBind-multimer/data/test/VHL_KRAS/VHL_KRAS_blocked.a3m'
    print('Using MSA 1:', TARGET_1_MSA)
    print('Using MSA 2:', TARGET_2_MSA)
    print('Using PAIRED MSA:', PAIRED_MSA)
    print('Using BLOCKED MSA:', BLOCKED_MSA)
else:
    TARGET_1_MSA = OUTDIR + TARGET_1_MSA
    TARGET_2_MSA = OUTDIR + TARGET_2_MSA
    PAIRED_MSA = OUTDIR + TARGET_1+'_'+TARGET_2 + '_paired.a3m'
    BLOCKED_MSA = OUTDIR + TARGET_1+'_'+TARGET_2 + '_blocked.a3m'

# Check TARGET 1 & 2 MSAs
msa1_missing = not os.path.exists(TARGET_1_MSA)
msa2_missing = not os.path.exists(TARGET_2_MSA)

if msa1_missing or msa2_missing:
    if msa1_missing: print(f"Can't find MSA for target 1: {TARGET_1_MSA}")
    if msa2_missing: print(f"Can't find MSA for target 2: {TARGET_2_MSA}")

    print("Generating missing MSAs using ColabFold...")
    !colabfold_batch "{single_queries_path}" "{job_name}" --msa-only

    if msa1_missing:
        TARGET_1_MSA = f"{job_name}/{TARGET_1}.a3m"
        print(f"New MSA 1 generated at: {TARGET_1_MSA}")
    if msa2_missing:
        TARGET_2_MSA = f"{job_name}/{TARGET_2}.a3m"
        print(f"New MSA 2 generated at: {TARGET_2_MSA}")

# CHECK AND GENERATE PAIRED/BLOCKED MSAs
paired_missing = not os.path.exists(PAIRED_MSA)
blocked_missing = not os.path.exists(BLOCKED_MSA)

if (paired_missing or blocked_missing):
    print("Generating Paired and Blocked MSAs...")
    max_gap = 0.9
    if paired_missing:
        !python /content/EvoBind-multimer/src/pair_msas.py \
            --a3m1 "{TARGET_1_MSA}" \
            --a3m2 "{TARGET_2_MSA}" \
            --max_gap_fraction {max_gap} \
            --outname "{PAIRED_MSA}"
        print(f"Generated paired MSA at: {PAIRED_MSA}")

    if blocked_missing:
        !python /content/EvoBind-multimer/src/block_msas.py \
            --a3m1 "{TARGET_1_MSA}" \
            --a3m2 "{TARGET_2_MSA}" \
            --max_gap_fraction {max_gap} \
            --outname "{BLOCKED_MSA}"
        print(f"Generated blocked MSA at: {BLOCKED_MSA}")

# process msas
from check_msa_colab import process_a3m
PROCESSED_MSA1 = TARGET_1_MSA.split('.')[0] + '_processed1.a3m'
process_a3m(TARGET_1_MSA, TARGET_1_SEQ, PROCESSED_MSA1)
TARGET_1_MSA = PROCESSED_MSA1

PROCESSED_MSA2 = TARGET_2_MSA.split('.')[0] + '_processed2.a3m'
process_a3m(TARGET_2_MSA, TARGET_2_SEQ, PROCESSED_MSA2)
TARGET_2_MSA = PROCESSED_MSA2

JOINT_SEQ = TARGET_1_SEQ + TARGET_2_SEQ
PROCESSED_PAIRED_MSA= PAIRED_MSA.split('.')[0] + '_processed.a3m'
process_a3m(PAIRED_MSA, JOINT_SEQ, PROCESSED_PAIRED_MSA)
PAIRED_MSA = PROCESSED_PAIRED_MSA

PROCESSED_BLOCKED_MSA= BLOCKED_MSA.split('.')[0] + '_processed.a3m'
process_a3m(BLOCKED_MSA, JOINT_SEQ, PROCESSED_BLOCKED_MSA)
BLOCKED_MSA = PROCESSED_BLOCKED_MSA

#MSAs = f"{PAIRED_MSA},{BLOCKED_MSA}"
MSAs = [PAIRED_MSA, BLOCKED_MSA]

#Check the start sequence length
if len(START_GLUE_SEQUENCE)>0:
  if len(START_GLUE_SEQUENCE)!=GLUE_LENGTH:
    print('The starting sequence length does not match the binder length')

#@markdown ----
#@markdown ### The target proteins are depicted in green (target 1) and cyan (target 2) cartoon format.

ModuleNotFoundError: No module named 'prepare_input_colab'

In [ ]:
#@markdown #Run *EvoBind multimer*

#@markdown Click play to design a peptide molecular glue.

#@markdown Each prediction (interation) takes around 3 minutes using the available T4 GPU. Relax and wait for your molecular glue.
#@markdown The run will continue where you left it if it was interrupted for some reason.

#@markdown The iteration, interface distance for target 1, interface distance for target 2, plDDT, delta COM, loss and best peptide sequence are displayed after each iteration.

#@markdown The AF2 params are fetched here (if they are not already downloaded).
import shutil
import collections
import os
import sys
import warnings
from Bio.PDB.PDBExceptions import PDBConstructionWarning
# Mute the discontinuous chain warnings
warnings.simplefilter('ignore', PDBConstructionWarning)
collections.Iterable = collections.abc.Iterable

# Set up clean directories for AlphaFold
DATADIR = "/content/scr/AF2/"
PARAMS = os.path.join(DATADIR, "params/")

# Download and extract params if they don't exist
if not os.path.exists(PARAMS):
  os.makedirs(PARAMS, exist_ok=True)
  print("Downloading AlphaFold parameters (this might take a minute)...")
  !wget -q https://storage.googleapis.com/alphafold/alphafold_params_2021-07-14.tar -O /content/alphafold_params_2021-07-14.tar

  print("Extracting parameters...")
  !tar -xf /content/alphafold_params_2021-07-14.tar -C {PARAMS}

  # Clean up the giant tar file to save Colab RAM/Disk space
  os.remove('/content/alphafold_params_2021-07-14.tar')
  print("AlphaFold parameters ready!")

sys.path.insert(0,'/content/EvoBind-multimer/src/AF2')
from mc_design_colab import main

MAX_RECYCLES = 8 #max_recycles (default=3)
MODEL_NAME = 'model_1' #model_1_ptm


# Run the main design function (Notice MSAS is all caps now!)
main(TARGET_1_FASTA, TARGET_2_FASTA, TARGET_RESIDUES_1, TARGET_RESIDUES_2,
     MSAs, GLUE_LENGTH,
     OUTDIR, NITER,
     [MODEL_NAME], MAX_RECYCLES, DATADIR,
     CYCLIC_OFFSET, START_GLUE_SEQUENCE)

/usr/local/lib/python3.12/dist-packages/Bio/Data/SCOPData.py:18: BiopythonDeprecationWarning: The 'Bio.Data.SCOPData' module will be deprecated in a future release of Biopython in favor of 'Bio.Data.PDBData.
  warnings.warn(


Results can be found in: /content/test_VHL_KRAS/design


In [ ]:
#@markdown #Analyse the results
#@markdown The TOP_FRACTION represents how many percent of the designs to select.

#@markdown Only the best model is visualised. As a rule of thumb, a **plDDT value above 85** represents a reliable glue.

#@markdown Click the DOWNLOAD box to download the top models and their sequences.

#@markdown Click the DOWNLOAD_START box to download the start model.
import pandas as pd
import os
TOP_FRACTION =  2#@param {type:"integer"}
TARGET_1_STYLE = "cartoon" #@param ["cartoon", "sphere", "stick"]
TARGET_2_STYLE = "cartoon" #@param ["cartoon", "sphere", "stick"]
MOL_GLUE_STYLE = "stick" #@param ["cartoon", "sphere", "stick"]
DOWNLOAD = False #@param {type:"boolean"}
DOWNLOAD_START = False #@param {type:"boolean"}
metrics = pd.read_csv('/content/8QU8/design/metrics.csv')
#Get top
metrics = metrics.sort_values(by='loss').reset_index()
n_select = int(TOP_FRACTION/100*len(metrics))
top_sel = metrics.loc[:n_select]

top_loss = top_sel.loss.values
top_sequence = top_sel.sequence.values
top_plddt = top_sel.plddt.values
top_models = top_sel.iteration.values

#Print
print('The best sequences, losses and plDDT values are:')
for i in range(len(top_loss)):
  print(top_sequence[i], top_loss[i], top_plddt[i])
#Vis
view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js',)
top_model = top_models[0]
if top_model=='init':
  model_path = OUTDIR+'design/unrelaxed_start.pdb'
else:
  model_path = OUTDIR+'/design/unrelaxed_'+str(top_model)+'.pdb'
view.addModel(open(model_path,'r').read(),'pdb')
view.setStyle({'chain':'A'},{TARGET_1_STYLE: {'color':'green'}})
view.setStyle({'chain':'B'},{TARGET_2_STYLE: {'color':'cyan'}})
view.setStyle({'chain':'C'},{MOL_GLUE_STYLE: {'color':'magenta'}})
view.zoomTo()
view.show()

#@title Download the results
import shutil
if not os.path.exists(OUTDIR+'best_models'):
  os.mkdir(OUTDIR+'best_models')

#Download
if DOWNLOAD==True:
  rank=1
  for model in top_models:
    if model=='init':
      model_path = OUTDIR+'/design/unrelaxed_start.pdb'
    else:
      model_path = OUTDIR+'/design/unrelaxed_'+str(top_model)+'.pdb'

    shutil.copy(model_path, OUTDIR+'best_models/rank_'+str(rank)+'.pdb')
    rank+=1

  for file in glob.glob(OUTDIR+'best_models/rank_*.pdb'):
    files.download(file)

  #Write a fasta file with the top seqs
  rank=1
  with open(OUTDIR+'best_models/top_seqs.fasta', 'w') as file:
    for seq in top_sequence:
      file.write('>rank_'+str(rank)+'\n')
      file.write(seq+'\n')
      rank+=1
  files.download(OUTDIR+'best_models/top_seqs.fasta')

if DOWNLOAD_START==True:
  files.download(OUTDIR+'/design/unrelaxed_start.pdb')
#@markdown ### Target 1 is depicted in green, target 2 in cyan and the molecular glue in magenta. Change the style above to view the design differently.
#@markdown ### Try the sphere representation to see how all atoms fit together.

The best sequences, losses and plDDT values are:
SKNQPPSP 0.1097939018603635 86.03032698508252


3Dmol.js failed to load for some reason. Please check your browser console for error messages.